# Multi-Model Demo
This notebook demonstrates calling three low-parameter models via OpenRouter and one high-parameter model via Groq concurrently.

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

openrouter_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

groq_client = AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# openai_client = AsyncOpenAI(
#     api_key=os.getenv("OPENAI_API_KEY"),
    
# )

In [2]:
async def get_response(client, model, prompt):
    try:
        response = await client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

async def main():
    prompt = "Explain the theory of relativity in one simple sentence."
    
    # Low parameter models on OpenRouter
    low_param_models = [
        "meta-llama/llama-4-maverick",
        "mistralai/mistral-small-2603",
        "google/gemma-4-26b-a4b-it:free"
    ]
    
    # High parameter model on Groq
    high_param_model = "llama-3.3-70b-versatile"
    
    print(f"Prompt: '{prompt}'\n")
    print("Calling models concurrently...")
    
    tasks = []
    for model in low_param_models:
        tasks.append(get_response(openrouter_client, model, prompt))
    
    tasks.append(get_response(groq_client, high_param_model, prompt))
    
    results = await asyncio.gather(*tasks)
    
    print("\n=== OpenRouter (Low Parameter Models) ===")
    for model, res in zip(low_param_models, results[:3]):
        print(f"\n[ {model} ]\n{res}")
        
    print("\n=== Groq (High Parameter Model) ===")
    print(f"\n[ {high_param_model} ]\n{results[3]}")

await main()

Prompt: 'Explain the theory of relativity in one simple sentence.'

Calling models concurrently...

=== OpenRouter (Low Parameter Models) ===

[ meta-llama/llama-4-maverick ]
The theory of relativity, developed by Albert Einstein, states that the laws of physics are the same for all observers, and that the passage of time and the length of objects can vary depending on their relative motion and position in a gravitational field.

[ mistralai/mistral-small-2603 ]
Error: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 52342. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 65428. To increase, visit https://openrouter.ai/settings/credits and up

## JSON Output Enforcement
This cell demonstrates how to enforce structured JSON outputs using `response_format={"type": "json_object"}`. It ensures the model responds exactly with the keys `"reasoning"` and `"answer"`.

In [5]:
async def get_json_response(client, model, prompt):
    system_prompt = """
You are a helpful assistant.

Respond with a valid JSON object containing exactly two keys:
{
  "explanation": "A brief explanation.",
  "answer": "The final concise answer."
}

Output ONLY valid JSON.
"""

    try:
        response = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"
        
async def json_demo():
    low_param_models = [
        "meta-llama/llama-4-maverick",
        "mistralai/mistral-small-2603",
        "google/gemma-4-26b-a4b-it:free"
    ]
    high_param_model = "llama-3.3-70b-versatile"
    prompt = "What is the capital of France and why is it important?"
    print(f"Prompt: '{prompt}'\n")
    
    tasks = []
    for model in low_param_models:
        tasks.append(get_json_response(openrouter_client, model, prompt))
    tasks.append(get_json_response(groq_client, high_param_model, prompt))
    
    results = await asyncio.gather(*tasks)
    
    print("=== OpenRouter (JSON Output) ===")
    for model, res in zip(low_param_models, results[:3]):
        print(f"\n[ {model} ]")
        try:
            parsed = json.loads(res)
            print(json.dumps(parsed, indent=2))
        except:
            print(res)
            
    print("\n=== Groq (JSON Output) ===")
    print(f"\n[ {high_param_model} ]")
    try:
        parsed = json.loads(results[3])
        print(json.dumps(parsed, indent=2))
    except:
        print(results[3])

await json_demo()

Prompt: 'What is the capital of France and why is it important?'

=== OpenRouter (JSON Output) ===

[ meta-llama/llama-4-maverick ]
{"explanation": "The capital of France is Paris, a city known for its rich history, art, fashion, and culture. It is considered one of the most important global centers for art, fashion, gastronomy, and culture. The city is home to numerous iconic landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum, making it a significant tourist destination and a symbol of French identity.", "answer": "Paris"}

[ mistralai/mistral-small-2603 ]
Error: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 52329. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens.

## BERTScore Consensus Evaluation
This cell evaluates the semantic similarity of the models' outputs using **BERTScore**. It compares the 3 low-parameter OpenRouter models against the high-parameter Groq model. If they all semantically agree (F1 Score > 0.85), it suppresses the reasoning and prints ONLY the final agreed answer.

In [4]:
import torch
from bert_score import score

async def evaluate_consensus():
    prompt = "What is the largest planet in our solar system and what is it mostly made of?"
    print(f"Evaluating Consensus for Prompt: '{prompt}'\n")
    
    low_param_models = [
        "meta-llama/llama-4-maverick",
        "mistralai/mistral-small-2603",
        "google/gemma-4-26b-a4b-it:free"
    ]
    high_param_model = "llama-3.3-70b-versatile"
    
    tasks = []
    for model in low_param_models:
        tasks.append(get_response(openrouter_client, model, prompt))
    tasks.append(get_response(groq_client, high_param_model, prompt))
    
    print("Generating responses...")
    results = await asyncio.gather(*tasks)
    
    groq_ans = results[3]
    low_param_ans = results[:3]
    
    print("Computing BERTScore...")
    # Compare each low-param model to the high-param model
    refs = [groq_ans] * 3
    P, R, F1 = score(low_param_ans, refs, lang="en", verbose=False)
    
    print("\nBERTScore F1 for each low-param model vs Groq model:")
    all_match = True
    for model, f1_score in zip(low_param_models, F1):
        score_val = f1_score.item()
        print(f"{model}: {score_val:.4f}")
        if score_val < 0.85:
            all_match = False
            
    if all_match:
        print("\n✅ CONSENSUS REACHED! All models strongly agree.")
        print(f"Final Answer:\n{groq_ans}")
    else:
        print("\n❌ CONSENSUS FAILED! The models disagree.")
        print("Outputs:")
        for m, r in zip(low_param_models + [high_param_model], results):
            print(f"[{m}]\n{r}\n")

await evaluate_consensus()

d:\SDE Project\Multi Agent Debate System\Modern-Debate-system-between-llms\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating Consensus for Prompt: 'What is the largest planet in our solar system and what is it mostly made of?'

Generating responses...
Computing BERTScore...


d:\SDE Project\Multi Agent Debate System\Modern-Debate-system-between-llms\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4769.91it/s]
[transfo


BERTScore F1 for each low-param model vs Groq model:
meta-llama/llama-4-maverick: 0.8863
mistralai/mistral-small-2603: 0.8743
google/gemma-4-26b-a4b-it:free: 0.8757

✅ CONSENSUS REACHED! All models strongly agree.
Final Answer:
The largest planet in our solar system is Jupiter. It is a gas giant, and its composition is mostly made up of:

1. Hydrogen (approximately 75%): Jupiter's atmosphere is primarily composed of hydrogen gas.
2. Helium (approximately 24%): Helium is the second most abundant element in Jupiter's atmosphere.
3. Methane, ammonia, and water (trace amounts): These compounds are present in smaller quantities and contribute to Jupiter's distinctive banded appearance and storm systems.

Jupiter's core is thought to be composed of denser materials, such as rocky or metallic compounds, but the exact composition is still a subject of scientific study and debate. However, the majority of the planet is made up of the hydrogen and helium gases mentioned above.
